In [30]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

reponse = llm.invoke("Dis bonjour en une phrase.")

print("RÉPONSE :", reponse.content)
print("RAISONNEMENT :", reponse.additional_kwargs.get("reasoning_content", "(aucun)")[:200])

RÉPONSE : Bonjour !
RAISONNEMENT : Okay, the user wants me to say "Dis bonjour en une phrase." which translates to "Say hello in one sentence." in French.

First, I need to make sure I understand the request correctly. They want a gree


In [31]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte."
)

In [32]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Je m'appelle Houssam"}]
})
print(response["messages"][-1].content)

Bonjour Houssam ! 😊


In [33]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Quel est mon prénom ?"}]
})
print(response["messages"][-1].content)

Je ne sais pas votre prénom.


In [34]:
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

In [35]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Je m'appelle Houssam"}]},
    config=config
)
print(res["messages"][-1].content)

Bonjour Houssam ! 😊


In [36]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config
)
print(res["messages"][-1].content)

Houssam 😊


In [37]:
config2 = {"configurable": {"thread_id": "2"}}

res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config2
)
print(res["messages"][-1].content)

Je ne sais pas votre prénom.


## Étape 2 : Tools personnalisés

In [38]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Donne la météo actuelle d'une ville."""
    print(f"[TOOL] get_weather appelé pour {location}")
    return f"Météo à {location} : ensoleillé, 32°C"

@tool
def get_employee_info(name: str) -> dict:
    """Donne le salaire et le poste d'un employé à partir de son prénom."""
    print(f"[TOOL] get_employee_info appelé pour {name}")
    return {"name": name, "salary": 12000, "job": "Développeur"}

@tool
def multiply(a: float, b: float) -> float:
    """Multiplie deux nombres."""
    print(f"[TOOL] multiply appelé avec a={a}, b={b}")
    return a * b

tools = [get_weather, get_employee_info, multiply]

In [39]:
print(get_weather.invoke({"location": "Marrakech"}))
print(multiply.invoke({"a": 6, "b": 7}))
print(get_weather.name, "-", get_weather.description)

[TOOL] get_weather appelé pour Marrakech
Météo à Marrakech : ensoleillé, 32°C
[TOOL] multiply appelé avec a=6.0, b=7.0
42.0
get_weather - Donne la météo actuelle d'une ville.


In [40]:
from langgraph.checkpoint.memory import InMemorySaver

agent_tools = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Tu es un assistant utile. Utilise les outils quand c'est nécessaire. Réponds en français.",
    checkpointer=InMemorySaver()
)

config_tools = {"configurable": {"thread_id": "tools-1"}}

In [41]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Casablanca ?"}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_weather appelé pour Casablanca
La météo à Casablanca est actuellement ensoleillée à 32°C.


In [42]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quel est le salaire de Mohamed ? Multiplie-le par 3."}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_employee_info appelé pour Mohamed
[TOOL] multiply appelé avec a=12000.0, b=3.0
Le salaire de Mohamed est de 12 000. Multiplié par 3, cela donne 36 000.


## Étape 3 : Tools prédéfinis

In [43]:
from ddgs import DDGS

for backend in ["duckduckgo", "brave", "bing", "google", "wikipedia"]:
    try:
        r = DDGS(timeout=20).text("capitale du Maroc", max_results=2, backend=backend)
        print(f"OK   {backend} : {len(r)} résultats - {r[0]['title'] if r else ''}")
    except Exception as e:
        print(f"FAIL {backend} : {str(e)[:80]}")

FAIL duckduckgo : No results found.
FAIL brave : No results found.
FAIL bing : No results found.
OK   google : 2 résultats - Maroc — WikipédiaRabat | Morocco’s Capital City, Map, & Historic Attractions ...Rabat - WikipediaQuelle est la capitale du Maroc ? Histoire et faits marquantsCapitale du Maroc : découvrir Rabat et les villes impérialesQuelle est la capitale du Maroc - moroccoanswers.com
FAIL wikipedia : No results found.


In [49]:
from ddgs import DDGS
from langchain.tools import tool

@tool
def ddg_bing(query: str) -> str:
    """Recherche sur le web (via DuckDuckGo/Bing) et renvoie les premiers résultats."""
    r = DDGS(timeout=20).text(query, max_results=3, backend="bing")
    return "\n".join(f"{x['title']} - {x['href']}\n{x['body']}" for x in r)

print(ddg_bing.invoke({"query": "Quelle est la capitale du Maroc ?"})[:400])

KeyboardInterrupt: 

> DuckDuckGo : requête expirée (timeout réseau). Voir Tavily ci-dessous.

In [50]:
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv(override=True)

tavily = TavilySearch(max_results=3)
resultat = tavily.invoke({"query": "Quelle est la capitale du Maroc ?"})

for r in resultat["results"]:
    print("-", r["title"])
    print("  ", r["url"])

- Quelle est la capitale du Maroc et pourquoi ce choix ?
   https://citopendia.fr/quelle-est-la-capitale-du-maroc-et-pourquoi-ce-choix
- Quelle est la capitale du Maroc ?
   https://darijaschool.com/quelle-est-la-capitale-du-maroc
- Quelle est la capitale du Népal ?
   https://memoriser-culture-generale.fr/quelle-est-la-capitale-du-nepal


In [51]:
from langchain_experimental.tools import PythonREPLTool

python_repl = PythonREPLTool()
print(python_repl.invoke("print(2 ** 10)"))

1024



In [52]:
from langgraph.checkpoint.memory import InMemorySaver

tools_pre = [tavily, python_repl]

agent_pre = create_agent(
    model=llm,
    tools=tools_pre,
    system_prompt=(
        "Tu es un assistant utile. Utilise l'outil de recherche web pour les "
        "informations récentes, et l'outil Python pour les calculs. Réponds en français."
    ),
    checkpointer=InMemorySaver()
)

config_pre = {"configurable": {"thread_id": "pre-1"}}

In [53]:
def poser(question):
    res = agent_pre.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config_pre
    )
    for m in res["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            print("[APPEL OUTIL]", tc["name"], tc["args"])
    print("\nRÉPONSE :", res["messages"][-1].content)

In [54]:
poser("Cherche sur le web quelle est la capitale du Maroc et donne-moi une source.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc', 'topic': 'general'}

RÉPONSE : La capitale du Maroc est **Rabat**. 

**Source** : [Wikipédia - Maroc](https://fr.wikipedia.org/wiki/Maroc)  
*Selon la Wikipédia, "La capitale administrative et politique du Maroc est Rabat tandis que la capitale économique et la plus grande ville du pays est Casablanca".*


In [ ]:
poser("Calcule la factorielle de 10 en utilisant Python.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc', 'topic': 'general'}
[APPEL OUTIL] Python_REPL {'query': 'import math; print(math.factorial(10))'}

RÉPONSE : La factorielle de 10 est **3628800**.


In [ ]:
poser("Calcule la factorielle de 10 en utilisant Python.")

[APPEL OUTIL] tavily_search {'query': 'capitale du Maroc', 'topic': 'general'}
[APPEL OUTIL] Python_REPL {'query': 'import math; print(math.factorial(10))'}

RÉPONSE : La factorielle de 10 est **3628800**.


## Étape 4 : Middlewares

### Middleware 1 : dynamic_model
Choisit le petit ou le grand modèle selon la longueur de la conversation.

In [47]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

petit_modele = ChatOllama(model="qwen3:1.7b", temperature=0, reasoning=True)
grand_modele = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

@wrap_model_call
def choix_modele(request: ModelRequest, handler) -> ModelResponse:
    """Petit modèle au début de la conversation, grand modèle ensuite."""
    nb = len(request.state["messages"])
    if nb > 4:
        modele, nom = grand_modele, "qwen3:4b"
    else:
        modele, nom = petit_modele, "qwen3:1.7b"
    print(f"[MIDDLEWARE] {nb} messages -> {nom}")
    return handler(request.override(model=modele))

In [55]:
agent_dyn = create_agent(
    model=grand_modele,
    tools=[get_weather, multiply],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    middleware=[choix_modele],
    checkpointer=InMemorySaver()
)

config_dyn = {"configurable": {"thread_id": "dyn-1"}}

In [56]:
for q in ["Bonjour !", "Quelle est la météo à Rabat ?", "Multiplie 8 par 7."]:
    res = agent_dyn.invoke({"messages": [{"role": "user", "content": q}]}, config=config_dyn)
    print("Q :", q)
    print("R :", res["messages"][-1].content, "\n")

[MIDDLEWARE] 1 messages -> qwen3:1.7b
Q : Bonjour !
R : Bonjour ! Je suis ici pour vous aider. Comment puis-je vous aider aujourd'hui ? 

[MIDDLEWARE] 3 messages -> qwen3:1.7b
[TOOL] get_weather appelé pour Rabat
[MIDDLEWARE] 5 messages -> qwen3:4b
Q : Quelle est la météo à Rabat ?
R : La météo à Rabat est ensoleillée à 32°C. 

[MIDDLEWARE] 7 messages -> qwen3:4b
[TOOL] multiply appelé avec a=8.0, b=7.0
[MIDDLEWARE] 9 messages -> qwen3:4b
Q : Multiplie 8 par 7.
R : 8 fois 7 égale 56. 



### Middleware 2 : dynamic_prompt
Adapte le prompt système selon le rôle de l'utilisateur (expert ou débutant).

In [57]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt, ModelRequest

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def prompt_selon_role(request: ModelRequest) -> str:
    """Génère le prompt système selon le rôle de l'utilisateur."""
    role = request.runtime.context.get("user_role", "user")
    base = "Tu es un assistant utile. Réponds en français."
    if role == "expert":
        prompt = f"{base} Donne une réponse technique et détaillée, en 4 phrases maximum."
    elif role == "beginner":
        prompt = f"{base} Explique simplement, sans jargon, avec une analogie, en 3 phrases maximum."
    else:
        prompt = base
    print(f"[PROMPT] rôle = {role}")
    return prompt

In [58]:
agent_prompt = create_agent(
    model=llm,
    tools=[],
    middleware=[prompt_selon_role],
    context_schema=Context,
)

In [59]:
question = "Explique ce qu'est le machine learning."

for role in ["expert", "beginner"]:
    res = agent_prompt.invoke(
        {"messages": [{"role": "user", "content": question}]},
        context={"user_role": role}
    )
    print(f"\n===== {role.upper()} =====")
    print(res["messages"][-1].content)

[PROMPT] rôle = expert

===== EXPERT =====
L'apprentissage automatique (ML) est une branche de l'intelligence artificielle permettant aux systèmes d'apprendre à partir de données pour identifier des motifs et effectuer des prédictions. Ces modèles utilisent des algorithmes statistiques pour analyser des ensembles de données complexes et extraire des patterns significatifs sans instructions explicites. L'objectif est de former un modèle capable d'améliorer sa précision et sa performance via des itérations d'entraînement, en s'adaptant aux nouvelles informations. Contrairement à la programmation classique, l'apprentissage automatique optimise automatiquement ses résultats en évitant la réécriture manuelle du code.
[PROMPT] rôle = beginner

===== BEGINNER =====
L'apprentissage automatique est une méthode où les ordinateurs apprennent à partir d'expériences sans être programmés à l'avance. C'est comme un enfant qui apprend à reconnaître des objets en regardant des exemples, sans avoir à êt

### Middleware 3 : gestion des erreurs d'outils

In [60]:
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@tool
def diviser(a: float, b: float) -> float:
    """Divise a par b."""
    return a / b

@wrap_tool_call
def gerer_erreurs_outils(request, handler):
    """Intercepte les erreurs d'outils et renvoie un message clair au modèle."""
    try:
        return handler(request)
    except Exception as e:
        print(f"[ERREUR OUTIL] {type(e).__name__} : {e}")
        return ToolMessage(
            content=f"Erreur de l'outil : {e}. Explique le problème à l'utilisateur.",
            tool_call_id=request.tool_call["id"]
        )

In [61]:
agent_err = create_agent(
    model=llm,
    tools=[diviser],
    system_prompt="Tu es un assistant utile. Utilise l'outil pour les divisions. Réponds en français, de façon courte.",
    middleware=[gerer_erreurs_outils],
)

In [62]:
res = agent_err.invoke(
    {"messages": [{"role": "user", "content": "Divise 10 par 0."}]}
)
print(res["messages"][-1].content)

[ERREUR OUTIL] ZeroDivisionError : float division by zero
Impossible de diviser par zéro.


### Middleware 4 : garde-fous (guardrails)

In [63]:
from typing import Any
from langchain.agents.middleware import before_agent, AgentState
from langgraph.runtime import Runtime

mots_interdits = ["hack", "exploit", "malware"]

@before_agent(can_jump_to=["end"])
def filtre_contenu(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Bloque les demandes contenant un mot interdit, avant tout traitement."""
    if not state["messages"]:
        return None

    premier = state["messages"][0]
    if premier.type != "human":
        return None

    texte = premier.content.lower()
    for mot in mots_interdits:
        if mot in texte:
            print(f"[GARDE-FOU] demande bloquée (mot : {mot})")
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "Je ne peux pas traiter cette demande. Reformulez-la, s'il vous plaît."
                }],
                "jump_to": "end"
            }
    return None

In [64]:
agent_garde = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    middleware=[filtre_contenu],
)

In [65]:
for q in ["Comment faire un hack de wifi ?", "Quelle est la météo à Marrakech ?"]:
    res = agent_garde.invoke({"messages": [{"role": "user", "content": q}]})
    print("Q :", q)
    print("R :", res["messages"][-1].content, "\n")

[GARDE-FOU] demande bloquée (mot : hack)
Q : Comment faire un hack de wifi ?
R : Je ne peux pas traiter cette demande. Reformulez-la, s'il vous plaît. 

[TOOL] get_weather appelé pour Marrakech
Q : Quelle est la météo à Marrakech ?
R : Marrakech : ensoleillé, 32°C 



### Middleware 5 : Human In The Loop

In [66]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def envoyer_email(adresse: str) -> str:
    """Envoie un e-mail à l'adresse donnée."""
    print(f"[ACTION RÉELLE] e-mail envoyé à {adresse}")
    return f"E-mail envoyé avec succès à {adresse}"

agent_hitl = create_agent(
    model=llm,
    tools=[envoyer_email],
    system_prompt="Tu es un assistant utile. Utilise l'outil pour envoyer des e-mails. Réponds en français, de façon courte.",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"envoyer_email": True}
        ),
    ],
    checkpointer=InMemorySaver(),
)

config_hitl = {"configurable": {"thread_id": "hitl-1"}}

In [67]:
result = agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Envoie un e-mail à med@gmail.com"}]},
    config=config_hitl
)
print(result["__interrupt__"])

[Interrupt(value={'action_requests': [{'name': 'envoyer_email', 'args': {'adresse': 'med@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: envoyer_email\nArgs: {'adresse': 'med@gmail.com'}"}], 'review_configs': [{'action_name': 'envoyer_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='1c69f560c7079260c2308a0a566fd223', response_schema=None)]


In [68]:
result = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config_hitl
)
print(result["messages"][-1].content)

[ACTION RÉELLE] e-mail envoyé à med@gmail.com
L'e-mail a été envoyé avec succès à med@gmail.com.


In [ ]:
config_rejet = {"configurable": {"thread_id": "hitl-2"}}

agent_hitl.invoke(
    {"messages": [{"role": "user", "content": "Envoie un e-mail à pirate@test.com"}]},
    config=config_rejet
)
result = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Envoi refusé par l'utilisateur."}]}),
    config=config_rejet
)
print(result["messages"][-1].content)